# Predict on external cohorts (ClinVar, gnomAD, GREGoR)

Runs `TrunCat` inference on all three cohorts with the final CatBoost model from notebook 03.

**Inputs per cohort** (all produced by the corrected feature scripts):
- `predict/<Cohort>_october26_final.csv`: Iman's base table after `fix_intron_flags.py` and `13_isoform_count_features.py`
- `predict/annotations/<prefix>_*`: codon optimality, readthrough, EJC occupancy, PTC AUG, conservation

**Pipeline per cohort:** load base table, merge annotations (`predict.merge_annotations`), `3UTR-No` fill, rebuild `EJC.downstream.cut` with the training bins, numeric `gnomAD_exome_ALL` (missing -> 0), guards, `clean_and_align`, predict, save.

**Guards that stop the run:** missing annotation files, non-unique `variantID`, changed row count after merge, duplicate `_x/_y` columns, any model feature missing after the merge.
**Guards that warn:** annotation coverage, corrected-intron fingerprint, categorical levels never seen in training.

Set `OUT_SUFFIX = ""` for the final run once the comparison in section 7 looks right.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier

# Notebook location: NMDpredictionmodel/Model/TrunCat/predict/
NOTEBOOK_DIR = Path.cwd().resolve()
TRUNCAT_DIR = NOTEBOOK_DIR.parent
REPO_ROOT = TRUNCAT_DIR.parent.parent
PREDICT_DIR = NOTEBOOK_DIR

assert (PREDICT_DIR / "predict.py").exists(), f"predict.py not found in {PREDICT_DIR}"
sys.path.insert(0, str(PREDICT_DIR))

import predict as _pm
for _name in ("merge_annotations", "clean_and_align", "predict_cohort",
              "sanity_check_predictions", "UTR3_MOTIF_PREFIXES"):
    assert hasattr(_pm, _name), f"predict.py is missing {_name}; wrong/old copy imported? Restart the kernel."
from predict import (merge_annotations, clean_and_align, predict_cohort,
                     sanity_check_predictions, _get_model_feature_info)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
print("predict.py loaded from:", _pm.__file__)
print("TRUNCAT_DIR:", TRUNCAT_DIR)

## 1. Configuration

In [ ]:
with open(TRUNCAT_DIR / "config" / "config.yaml") as f:
    cfg = yaml.safe_load(f)
DATA_CFG, OUTPUT_CFG = cfg["data"], cfg["output"]

def _resolve(rel):
    for base in (TRUNCAT_DIR, REPO_ROOT):
        p = base / rel
        if p.exists():
            return p
    return TRUNCAT_DIR / rel

FEATURE_LIST_PATH = TRUNCAT_DIR / DATA_CFG["feature_list"]
FINAL_MODEL_CBM = TRUNCAT_DIR / OUTPUT_CFG["final_model_cbm"]
CLEANED_PATH = _resolve(DATA_CFG["cleaned"])          # training reference for categorical levels
VIZ_SUMMARY_PATH = TRUNCAT_DIR / "results" / "figures" / "visualizations_cv" / "viz_summary.json"
TRAINING_MEDIANS_PATH = PREDICT_DIR / "training_medians.json"
ANNOT_DIR = PREDICT_DIR / "annotations"

# Suffix on every output so earlier predictions stay available for section 7. "" for the final run.
OUT_SUFFIX = ""

# fix_intron_flags.py writes NA for "no 3'UTR intron"; the TOPMed training table uses this label.
THREEUTR_NO_LABEL = "3UTR-No"

# Stop if the model expects a feature that is absent after the merge
REQUIRE_ALL_FEATURES = True

EXTRA_ID_COLS = ["GENE_ID", "hgnc_symbol", "CHROM", "POS", "REF_ALLELE", "ALT_ALLELE"]

# key -> (base table prefix, annotation file prefix, output name stem)
COHORT_DEFS = {
    "clinvar": ("clinVar", "clinvar", "clinvar"),
    "gnomAD":  ("gnomAD",  "gnomAD",  "gnomAD"),
    "gregor":  ("GREGoR",  "GREGoR",  "gregor"),
}
RUN = ["clinvar", "gnomAD", "gregor"]     # edit to run a subset

COHORTS = {}
for key, (base, ann, stem) in COHORT_DEFS.items():
    COHORTS[key] = {
        "input": PREDICT_DIR / f"{base}_october26_final.csv",
        "annotations": {
            "codon_opt":    ANNOT_DIR / f"{ann}_codon_optimality.tsv",
            "readthrough":  ANNOT_DIR / f"{ann}_readthrough.csv",
            "ejc":          ANNOT_DIR / f"{ann}_ejc_occupancy.tsv",
            "ptc_aug":      ANNOT_DIR / f"{ann}_ptc_aug.tsv",
            "conservation": ANNOT_DIR / f"{ann}_variants_with_conservation_medians.csv",
        },
        "output":     PREDICT_DIR / f"{stem}_predictions{OUT_SUFFIX}.csv",
        "audit_json": PREDICT_DIR / f"{stem}_audit{OUT_SUFFIX}.json",
        "plot":       PREDICT_DIR / f"{stem}_prediction_summary{OUT_SUFFIX}.png",
    }

print(f"{'cohort':8s} input / annotation files present")
for key in RUN:
    c = COHORTS[key]
    ok = [c["input"].exists()] + [p.exists() for p in c["annotations"].values()]
    print(f"{key:8s} {sum(ok)}/{len(ok)}  {c['input'].name}")

## 2. Load model, threshold, training medians, training categorical levels

In [ ]:
assert FINAL_MODEL_CBM.exists(), f"Model not found: {FINAL_MODEL_CBM}"
model = CatBoostClassifier()
model.load_model(str(FINAL_MODEL_CBM))
model_features, model_cat_features = _get_model_feature_info(model)
print(f"Model: {FINAL_MODEL_CBM.name} | {len(model_features)} features ({len(model_cat_features)} categorical)")

assert FEATURE_LIST_PATH.exists(), f"feature list not found: {FEATURE_LIST_PATH}"
_fl = pd.read_csv(FEATURE_LIST_PATH)["feature"].tolist()
assert set(_fl) == set(model_features), (
    f"Model ({len(model_features)}) and final_feature_list.csv ({len(_fl)}) disagree: "
    f"{sorted(set(_fl) ^ set(model_features))[:10]} ... stale model or feature list?")
print(f"Matches final_feature_list.csv; same order: {_fl == model_features}")

assert VIZ_SUMMARY_PATH.exists(), f"viz_summary.json not found: {VIZ_SUMMARY_PATH}"
with open(VIZ_SUMMARY_PATH) as f:
    viz_summary = json.load(f)
THRESHOLD = float(viz_summary["threshold_used"])
print(f"Threshold: {THRESHOLD:.4f} | OOF AUC in viz_summary: {viz_summary.get('oof_auc', 'n/a')}")
print("  -> this AUC should equal the AUC you report for the final run of notebook 03")

assert TRAINING_MEDIANS_PATH.exists(), f"training medians not found: {TRAINING_MEDIANS_PATH}"
with open(TRAINING_MEDIANS_PATH) as f:
    training_medians = json.load(f)
print(f"Training medians: { {k: round(float(v), 4) for k, v in training_medians.items()} }")

# Categorical levels seen in training (post-cleaning), for the unseen-level guard
train_levels = {}
if CLEANED_PATH.exists():
    _hdr = pd.read_csv(CLEANED_PATH, nrows=0).columns
    _cats = [c for c in model_cat_features if c in _hdr]
    _tr = pd.read_csv(CLEANED_PATH, usecols=_cats, low_memory=False)
    train_levels = {c: set(_tr[c].astype(str).unique()) for c in _cats}
    del _tr
    print(f"Training categorical levels loaded for {len(train_levels)}/{len(model_cat_features)} features from {CLEANED_PATH.name}")
else:
    print(f"WARNING: {CLEANED_PATH} not found; unseen-level guard disabled")

## 3. Helper checks

In [ ]:
ANNOT_COVERAGE_COLS = {
    "codon":        ["CodonOptimalityFraction_CDS"],
    "readthrough":  ["readthrough_score_hek293t"],
    "ejc":          ["ejc_count_in_window"],
    "aug":          ["aug_distance_category", "kozak_strength", "aug_frame_status"],
    "conservation": ["phastcons_tx_whole_median", "phylop_tx_whole_median"],
}

def corrected_fingerprint(df, header):
    out = []
    if "threeUTR.introns" in df.columns:
        v = df["threeUTR.introns"].astype(str).str.contains("There is", na=False).mean()
        out.append(("3'UTR intron rate (corrected ~2-3%, old ~24-28%)", f"{v:.1%}", v < 0.10))
    n_rbp = len({c.split(".", 1)[1] for c in header if c.startswith("utr3_all.")})
    out.append(("utr3_all RBP columns (corrected 159, old 125)", str(n_rbp), n_rbp >= 150))
    return out

def coverage_report(df):
    print("\nAnnotation coverage after merge (share of NaN):")
    for src, cols in ANNOT_COVERAGE_COLS.items():
        for c in cols:
            if c not in df.columns:
                print(f"  WARN {src:12s} {c}: column absent")
                continue
            na = df[c].isna().mean()
            print(f"  {'WARN' if na > 0.10 else 'OK  '} {src:12s} {c}: {na:.1%}")

# Training bins for EJC.downstream.cut. Iman's ClinVar/gnomAD tables use an older scheme
# (0,1,2,3,>3); the TOPMed table and GREGoR use these. Rebuilt from the raw count.
EJC_BINS = [-np.inf, 2, 5, 9, 14, np.inf]
EJC_LABELS = ["0-2", "3-5", "6-9", "10-14", ">14"]

def ejc_cut(counts):
    return pd.cut(pd.to_numeric(counts, errors="coerce"), bins=EJC_BINS, labels=EJC_LABELS).astype(object)

def verify_ejc_bins_against_training():
    path = _resolve(DATA_CFG["merged"])
    if not path.exists():
        print(f"WARNING: {path} not found; EJC bin check skipped")
        return
    hdr = pd.read_csv(path, nrows=0).columns
    if not {"EJC.downstream", "EJC.downstream.cut"} <= set(hdr):
        print("WARNING: training table lacks EJC.downstream / EJC.downstream.cut; EJC bin check skipped")
        return
    t = pd.read_csv(path, usecols=["EJC.downstream", "EJC.downstream.cut"], low_memory=False)
    rebuilt = ejc_cut(t["EJC.downstream"]).astype(str)
    stored = t["EJC.downstream.cut"].astype(str)
    mism = int((rebuilt != stored).sum())
    assert mism == 0, f"EJC bins do not reproduce training: {mism} mismatches. Fix EJC_BINS."
    print(f"EJC.downstream.cut bins reproduce the training table exactly ({len(t):,} rows)")

verify_ejc_bins_against_training()

def unseen_level_report(X, name):
    print(f"\nCategorical levels not seen in training ({name}):")
    found = False
    for c in model_cat_features:
        if c not in train_levels or c not in X.columns:
            continue
        vc = X[c].astype(str).value_counts()
        bad = vc[~vc.index.isin(train_levels[c])]
        if len(bad):
            found = True
            print(f"  WARN {c}: {bad.head(5).to_dict()}")
    if not found:
        print("  OK   none")

## 4. Run predictions

`variantID` must be unique in the base table and in every annotation file, or the merge would change the row count.
Only the columns the model needs (plus IDs) are loaded, which keeps the gnomAD table manageable.

In [ ]:
results = {}

for key in RUN:
    cohort = COHORTS[key]
    print("\n" + "=" * 80)
    print(f"COHORT: {key}")
    print("=" * 80)

    # ---- files present
    assert cohort["input"].exists(), f"input not found: {cohort['input']}"
    missing_files = [k for k, p in cohort["annotations"].items() if not p.exists()]
    assert not missing_files, f"annotation files missing for {key}: {missing_files}"

    # ---- load only needed columns
    header = pd.read_csv(cohort["input"], nrows=0).columns.tolist()
    assert "variantID" in header, f"{cohort['input'].name} has no variantID column"
    needed = set(model_features) | {"variantID", "loefu_cat", "LOEUF_cat", "threeUTR.introns", "EJC.downstream", "gnomAD_exome_ALL"} | set(EXTRA_ID_COLS)
    keep = [c for c in header if c in needed]
    df = pd.read_csv(cohort["input"], usecols=keep, low_memory=False)
    n_in = len(df)
    print(f"Loaded {cohort['input'].name}: {n_in:,} rows, {len(keep)}/{len(header)} columns kept")
    assert df["variantID"].is_unique, f"{key}: variantID is not unique in the base table"

    # ---- fingerprint of corrected intron flags (before the fill)
    print("\nCorrected-features fingerprint:")
    for label, value, ok in corrected_fingerprint(df, header):
        print(f"  {'OK  ' if ok else 'WARN'} {label}: {value}")

    # ---- merge annotations
    print("\nMerging annotations:")
    df = merge_annotations(
        df,
        path_codon_opt=cohort["annotations"]["codon_opt"],
        path_readthrough=cohort["annotations"]["readthrough"],
        path_ejc=cohort["annotations"]["ejc"],
        path_ptc_aug=cohort["annotations"]["ptc_aug"],
        path_conservation=cohort["annotations"]["conservation"],
    )
    assert len(df) == n_in, f"{key}: merge changed row count {n_in} -> {len(df)} (duplicate variantIDs in an annotation file?)"
    dups = [c for c in df.columns if c.endswith(("_x", "_y"))]
    assert not dups, f"{key}: duplicate columns after merge: {dups[:10]}"
    coverage_report(df)

    # ---- fix_intron_flags.py writes NA for "no 3'UTR intron"; training uses the label
    if "threeUTR.introns" in df.columns:
        n_na = int(df["threeUTR.introns"].isna().sum())
        df["threeUTR.introns"] = df["threeUTR.introns"].fillna(THREEUTR_NO_LABEL)
        print(f"\nthreeUTR.introns: filled {n_na:,} NA with '{THREEUTR_NO_LABEL}'")

    # ---- EJC.downstream.cut: rebuild with the training bins from the raw count
    if "EJC.downstream" in df.columns:
        new_cut = ejc_cut(df["EJC.downstream"])
        if "EJC.downstream.cut" in df.columns:
            diff = (df["EJC.downstream.cut"].astype(str) != new_cut.astype(str)).mean()
            print(f"EJC.downstream.cut: {diff:.1%} of rows differ from the table's own labels (rebuilt with training bins)")
        df["EJC.downstream.cut"] = new_cut

    # ---- gnomAD_exome_ALL: cohorts code "not found" as '.', training zero-filled missing values
    if "gnomAD_exome_ALL" in df.columns:
        num = pd.to_numeric(df["gnomAD_exome_ALL"], errors="coerce")
        n_conv = int(num.isna().sum())
        df["gnomAD_exome_ALL"] = num.fillna(0)
        print(f"gnomAD_exome_ALL: {n_conv:,} non-numeric/NaN values -> 0 ({n_conv / len(df):.1%})")

    # ---- stash IDs
    variant_ids = df["variantID"].copy()
    id_cols = [c for c in EXTRA_ID_COLS if c in df.columns]
    skipped = [c for c in EXTRA_ID_COLS if c not in df.columns]
    if skipped:
        print(f"extra id columns not found (skipped): {skipped}")
    extra_ids_df = df[id_cols].copy() if id_cols else None

    # ---- clean, align, audit
    X_aligned, audit = clean_and_align(df, model, training_medians)
    audit.print()
    if REQUIRE_ALL_FEATURES:
        assert not audit.missing, (
            f"{key}: {len(audit.missing)} model features missing after merge: {audit.missing[:15]} ... "
            "Predictions would run with these blank. Fix the inputs (or set REQUIRE_ALL_FEATURES = False).")
    unseen_level_report(X_aligned, key)

    cohort["audit_json"].parent.mkdir(parents=True, exist_ok=True)
    with open(cohort["audit_json"], "w") as f:
        json.dump(audit.to_dict(), f, indent=2)

    # ---- predict + save
    print(f"\nRunning predictions on {len(X_aligned):,} variants...")
    pred_df = predict_cohort(X_aligned, model, variant_ids, threshold=THRESHOLD, extra_id_cols=extra_ids_df)
    sanity_check_predictions(pred_df, THRESHOLD)
    pred_df.to_csv(cohort["output"], index=False)
    print(f"\nSaved: {cohort['output']}")

    results[key] = {"predictions": pred_df, "input_rows": n_in, "audit": audit}
    del df, X_aligned

## 5. Final checklist

Every output has the required columns, valid probabilities, unique `variantID`s, and the same number of rows as the input table.

In [ ]:
required = ["variantID", "escape_probability", "predicted_class", "predicted_label", "threshold_used"]
all_pass = bool(results)
for key, r in results.items():
    p = r["predictions"]
    checks = {
        "required columns":       all(c in p.columns for c in required),
        "row count matches input": len(p) == r["input_rows"],
        "probabilities in [0,1]": bool(p["escape_probability"].between(0, 1).all()),
        "no NaN probabilities":   bool(not p["escape_probability"].isna().any()),
        "classes in {0,1}":       bool(set(p["predicted_class"].unique()) <= {0, 1}),
        "labels valid":           bool(set(p["predicted_label"].unique()) <= {"escape", "NMD"}),
        "variantIDs unique":      bool(p["variantID"].is_unique),
        "no missing model features": len(r["audit"].missing) == 0,
    }
    ok = all(checks.values())
    all_pass &= ok
    print(f"[{'PASS' if ok else 'FAIL'}] {key}: " + ", ".join(k for k, v in checks.items() if not v) if not ok
          else f"[PASS] {key}: {len(p):,} variants | escape {p['predicted_class'].mean():.1%}")
print("\n" + ("ALL COHORTS PASSED" if all_pass else "ONE OR MORE COHORTS FAILED - investigate above"))

## 6. Summary figures

Probability distribution with the threshold, and class counts, for each cohort.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

PREDICTION_COLORS = {'NMD': '#FF6B6B', 'escape': '#4ECDC4'}
SENSITIVE_LABEL = 'Sensitive'
ESCAPE_LABEL = 'Escape'
COHORT_DISPLAY = {'clinvar': 'ClinVar', 'gnomAD': 'gnomAD', 'gregor': 'GREGoR'}


def assign_confidence(prob, threshold, low_band=0.08, high_band=0.15):
    """Distance-from-threshold confidence binning. Same heuristic bands as the earlier figures."""
    distance = abs(prob - threshold)
    if distance > high_band:
        return 'high'
    elif distance > low_band:
        return 'medium'
    else:
        return 'low'


def plot_cohort_summary(pred_df, threshold, cohort_name, save_path=None):
    """4-panel summary: histogram, confidence bars, boxplot, pie."""
    df = pred_df.copy()
    df['display_label'] = df['predicted_label'].map({'NMD': SENSITIVE_LABEL, 'escape': ESCAPE_LABEL})
    df['confidence'] = df['escape_probability'].apply(lambda p: assign_confidence(p, threshold))

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'{cohort_name}  —  n = {len(df):,} variants', fontsize=18, fontweight='bold', y=1.00)

    # Panel 1: histogram
    ax = axes[0, 0]
    ax.hist(df['escape_probability'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(threshold, color='red', linestyle='--', linewidth=2.5, label=f'Optimal threshold ({threshold:.3f})')
    ax.set_xlabel('Escape Probability', fontsize=14, fontweight='bold')
    ax.set_ylabel('Count', fontsize=14, fontweight='bold')
    ax.set_title('Distribution of NMD Escape Probabilities', fontsize=16, fontweight='bold')
    ax.legend(fontsize=14)
    ax.tick_params(axis='both', labelsize=14)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    # Panel 2: predictions by confidence level
    ax = axes[0, 1]
    conf_pred = (df.groupby(['confidence', 'display_label']).size()
                   .unstack(fill_value=0).reindex(['high', 'medium', 'low'], fill_value=0))
    for col in [SENSITIVE_LABEL, ESCAPE_LABEL]:
        if col not in conf_pred.columns:
            conf_pred[col] = 0
    conf_pred = conf_pred[[SENSITIVE_LABEL, ESCAPE_LABEL]]
    conf_pred_pct = (conf_pred / len(df)) * 100

    x = np.arange(len(conf_pred_pct.index))
    width = 0.35
    bars_sens = ax.bar(x - width / 2, conf_pred_pct[SENSITIVE_LABEL].values, width, label=SENSITIVE_LABEL,
                       color=PREDICTION_COLORS['NMD'], alpha=0.85, edgecolor='black', linewidth=0.5)
    bars_esc = ax.bar(x + width / 2, conf_pred_pct[ESCAPE_LABEL].values, width, label=ESCAPE_LABEL,
                      color=PREDICTION_COLORS['escape'], alpha=0.85, edgecolor='black', linewidth=0.5)
    for bars, col in [(bars_sens, SENSITIVE_LABEL), (bars_esc, ESCAPE_LABEL)]:
        for rect, conf_lvl in zip(bars, conf_pred_pct.index):
            pct = rect.get_height()
            n = int(conf_pred.loc[conf_lvl, col])
            if pct > 0:
                ax.text(rect.get_x() + rect.get_width() / 2, pct, f'{pct:.1f}%\n(n={n:,})',
                        ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([s.capitalize() for s in conf_pred_pct.index])
    ax.set_xlabel('Confidence Level', fontsize=14, fontweight='bold')
    ax.set_ylabel('Percentage of Total', fontsize=14, fontweight='bold')
    ax.set_title('Predictions by Confidence Level', fontsize=16, fontweight='bold')
    ax.legend(fontsize=14)
    ax.tick_params(axis='both', labelsize=14)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    # Panel 3: boxplot by predicted class
    ax = axes[1, 0]
    sens_probs = df.loc[df['display_label'] == SENSITIVE_LABEL, 'escape_probability']
    esc_probs = df.loc[df['display_label'] == ESCAPE_LABEL, 'escape_probability']
    try:   # matplotlib >= 3.9 renamed `labels` to `tick_labels`
        bp = ax.boxplot([sens_probs, esc_probs], tick_labels=[SENSITIVE_LABEL, ESCAPE_LABEL],
                        patch_artist=True, widths=0.5)
    except TypeError:
        bp = ax.boxplot([sens_probs, esc_probs], labels=[SENSITIVE_LABEL, ESCAPE_LABEL],
                        patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor(PREDICTION_COLORS['NMD']); bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor(PREDICTION_COLORS['escape']); bp['boxes'][1].set_alpha(0.7)
    for element in ['whiskers', 'fliers', 'means', 'medians', 'caps']:
        if element in bp:
            plt.setp(bp[element], linewidth=2)
    ax.axhline(threshold, color='red', linestyle='--', linewidth=2.5, alpha=0.8, label=f'Threshold ({threshold:.3f})')
    ax.set_ylabel('Escape Probability', fontsize=14, fontweight='bold')
    ax.set_title('Probability Distribution by Predicted Class', fontsize=16, fontweight='bold')
    ax.legend(fontsize=14)
    ax.tick_params(axis='both', labelsize=14)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    # Panel 4: pie chart
    ax = axes[1, 1]
    pred_counts = df['display_label'].value_counts()
    pie_colors = [PREDICTION_COLORS['NMD'] if lbl == SENSITIVE_LABEL else PREDICTION_COLORS['escape']
                  for lbl in pred_counts.index]
    wedges, texts, autotexts = ax.pie(pred_counts.values, labels=pred_counts.index, autopct='%1.1f%%',
                                      colors=pie_colors, startangle=90,
                                      textprops={'fontsize': 14, 'fontweight': 'bold'})
    for autotext in autotexts:
        autotext.set_color('white'); autotext.set_fontsize(14); autotext.set_fontweight('bold')
    ax.set_title('NMD Prediction Distribution', fontsize=16, fontweight='bold')

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f'  saved: {save_path}')
    plt.show()
    return fig


for key, r in results.items():
    plot_cohort_summary(
        pred_df=r['predictions'],
        threshold=THRESHOLD,
        cohort_name=COHORT_DISPLAY.get(key, key),
        save_path=COHORTS[key]['plot'],
    )

## 7. Old vs corrected predictions

Compares each cohort with the earlier predictions file of the same name (without the suffix), if it exists.
Expect strong rank correlation but not identity, since the features were corrected.
Very low correlation or a large share of class flips points to a misalignment worth investigating.

In [ ]:
from scipy.stats import spearmanr

for key, r in results.items():
    if not OUT_SUFFIX:
        print(f"{key}: OUT_SUFFIX is empty, so the earlier file was overwritten; nothing to compare.")
        continue
    new_path = COHORTS[key]["output"]
    old_path = new_path.with_name(new_path.name.replace(f"{OUT_SUFFIX}.csv", ".csv"))
    if not old_path.exists():
        print(f"{key}: no earlier predictions at {old_path.name}")
        continue
    old = pd.read_csv(old_path, usecols=["variantID", "escape_probability", "predicted_class"])
    new = r["predictions"][["variantID", "escape_probability", "predicted_class"]]
    m = new.merge(old, on="variantID", suffixes=("_new", "_old"))
    print(f"\n{key}: {len(m):,} shared variants ({len(new):,} new / {len(old):,} old)")
    if len(m) < 10:
        print("  too few shared variants to compare")
        continue
    rho = spearmanr(m.escape_probability_new, m.escape_probability_old)[0]
    flips = (m.predicted_class_new != m.predicted_class_old).mean()
    dmean = (m.escape_probability_new - m.escape_probability_old).abs().mean()
    print(f"  Spearman = {rho:.3f} | class flips = {flips:.1%} | mean |delta p| = {dmean:.3f}")